# Session 16 · scikit-learn fundamentals

Work top to bottom and run each cell with **Shift+Enter**. Cells under a **Your turn** heading contain `...` placeholders: replace them, run the cell, then run the check beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook. Solutions are in the matching `_solutions` notebook.

The data is simulated inside the notebook from a **known** model, so you can see whether each method recovers the truth.

| tidymodels | scikit-learn |
| --- | --- |
| `initial_split(df, strata = y)` | `train_test_split(X, y, stratify=y)` |
| `recipe(y ~ ., df) %>% step_dummy() %>% step_normalize()` | `ColumnTransformer([...])` with `OneHotEncoder` and `StandardScaler` |
| `workflow() %>% add_recipe() %>% add_model()` | `Pipeline([("prep", ...), ("model", ...)])` |
| `fit(wf, train)` | `pipe.fit(X_train, y_train)` |
| `predict(fit, test, type = "prob")` | `pipe.predict_proba(X_test)[:, 1]` |
| `vfold_cv()` + `fit_resamples()` | `cross_val_score(pipe, X, y, cv=5)` |

In [1]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(16)
n = 5_000
members = pd.DataFrame({
    "age": rng.integers(18, 80, n),
    "tenure_years": rng.uniform(0, 10, n).round(1),
    "tier": rng.choice(["basic", "mid", "top"], n, p=[0.5, 0.35, 0.15]),
    "channel": rng.choice(["direct", "broker", "online"], n, p=[0.5, 0.2, 0.3]),
})

# The truth: log-odds of churning in the year
true_logit = (
    -0.8
    - 0.02 * (members["age"] - 45)
    - 0.15 * members["tenure_years"]
    + members["tier"].map({"basic": 0, "mid": -0.3, "top": -0.7})
    + members["channel"].map({"direct": 0, "broker": 0.3, "online": 0.5})
)
members["churned"] = (rng.random(n) < 1 / (1 + np.exp(-true_logit))).astype(int)

# A second outcome for Poisson models: claims in the year, given exposure
members["exposure"] = rng.uniform(0.2, 1.0, n).round(2)
true_log_rate = (np.log(0.8) + 0.015 * (members["age"] - 45)
                 + members["tier"].map({"basic": 0, "mid": 0.2, "top": 0.4}))
members["claims"] = rng.poisson(members["exposure"] * np.exp(true_log_rate))

print(f"{members['churned'].mean():.1%} churned")
members.head()

18.9% churned


,age,tenure_years,tier,channel,churned,exposure,claims
0,51,7.0,top,broker,0,0.38,1
1,53,3.0,basic,online,0,0.45,0
2,69,4.6,basic,online,0,0.67,2
3,44,0.8,mid,broker,0,0.87,4
4,65,7.8,basic,direct,0,0.88,3


## 1. The estimator pattern

Every scikit-learn model works the same way: create it, `.fit(X, y)`, then `.predict(X)` or `.predict_proba(X)`. `X` is a DataFrame of features, and `y` is the outcome.

### Your turn 1.1

Split the data 75/25 into training and test sets. Stratify on the outcome, so both sets have the same churn rate, and use `random_state=16` so the split is repeatable.

In [3]:
from sklearn.model_selection import train_test_split

X = members[["age", "tenure_years", "tier", "channel"]]
y = members["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)

In [4]:
check("1.1 test size", lambda: len(X_test), 1250)
check("1.1 stratified", lambda: abs(y_train.mean() - y_test.mean()) < 0.005)

PASS     1.1 test size
PASS     1.1 stratified


### Your turn 1.2

Fit a `LogisticRegression` on the two numeric columns only, predict the churn probability for the test set, and score it with `roc_auc_score`. `predict_proba` returns one column per class; `[:, 1]` takes the probability of churning.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

numeric = ["age", "tenure_years"]
model_num = LogisticRegression(solver="liblinear")
model_num.fit(X_train[numeric], y_train)
p_num = model_num.predict_proba(X_test[numeric])[:, 1]
auc_num = roc_auc_score(y_test, p_num)
auc_num

0.6563148516926947

In [6]:
check("1.2 better than chance", lambda: auc_num > 0.6)

PASS     1.2 better than chance


## 2. Pipelines: recipes and workflows

A `ColumnTransformer` says what to do to which columns, like a recipe. A `Pipeline` chains it with the model, like a workflow. Fitting the pipeline learns the scaling and the encoding from the **training data only**, which prevents leakage.

**Choose your reference levels explicitly.** `OneHotEncoder(drop="first")` drops the first level **alphabetically**, which for `channel` is "broker", not "direct". Passing `categories=` in your own order is the equivalent of setting factor levels in R.

**scikit-learn regularises by default.** `LogisticRegression()` applies an L2 (ridge) penalty unless you say otherwise. R's `glm()` does not. When you want coefficients you can compare with R or with the truth, use `penalty=None`.

### Your turn 2.1

Build `pipe`:

- `prep`: a `ColumnTransformer` that applies `StandardScaler()` to the numeric columns and `OneHotEncoder(categories=[["basic", "mid", "top"], ["direct", "broker", "online"]], drop="first")` to `tier` and `channel`
- `model`: `LogisticRegression(penalty=None, max_iter=1000)`

Fit it on the training data and set `auc_full` to its test AUC.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

prep = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(categories = [["basic", "mid", "top"], ["direct", "broker", "online"]], drop='first'), ["tier", "channel"]),
])

pipe = Pipeline([("prep", prep), ("model", LogisticRegression(penalty=None, max_iter=1000))])
pipe.fit(X_train, y_train)
auc_full = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
auc_full

/Users/benlenepveu/code/python-learning/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


0.6635929147888062

In [9]:
check("2.1 categories add information", lambda: auc_full > auc_num)

PASS     2.1 categories add information


## 3. Did it recover the truth?

`get_feature_names_out()` names each column the pipeline produced, so you can line the coefficients up with them. The categorical coefficients are on the same log-odds scale as the truth. The numeric ones are per **standard deviation**, because of the scaling, so divide by the scaler's `scale_` to get per-unit effects.

### Your turn 3.1

Build `coefs`: a Series of the model's coefficients, indexed by the feature names. Then compare `coefs["cat__tier_top"]` with the true -0.7, and `coefs["cat__channel_online"]` with the true 0.5.

In [11]:
names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(pipe.named_steps["model"].coef_[0], index=names)
scale = pipe.named_steps["prep"].named_transformers_["num"].scale_
print("Age per year:", round(coefs["num__age"] / scale[0], 2))
coefs.round(2)

Age per year: -0.02


num__age              -0.39
num__tenure_years     -0.44
cat__tier_mid         -0.29
cat__tier_top         -0.70
cat__channel_broker    0.54
cat__channel_online    0.65
dtype: float64

In [12]:
check("3.1 tier top near -0.7", lambda: abs(coefs["cat__tier_top"] - (-0.7)) < 0.3)
check("3.1 online near 0.5", lambda: abs(coefs["cat__channel_online"] - 0.5) < 0.3)

PASS     3.1 tier top near -0.7
PASS     3.1 online near 0.5


## 4. Cross-validation

`cross_val_score` refits the whole pipeline on each fold, preprocessing included, and returns one score per fold.

### Your turn 4.1

Set `cv_auc` to the 5-fold cross-validated AUC scores for `pipe` on all of `X` and `y` (`scoring="roc_auc"`).

In [15]:
from sklearn.model_selection import cross_val_score

cv_auc = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")

cv_auc.round(3), cv_auc.mean().round(3)

/Users/benlenepveu/code/python-learning/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/benlenepveu/code/python-learning/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.wa

(array([0.676, 0.649, 0.688, 0.687, 0.684]), np.float64(0.677))

In [16]:
check("4.1 five folds", lambda: len(cv_auc), 5)
check("4.1 stable", lambda: cv_auc.std() < 0.05)

PASS     4.1 five folds
PASS     4.1 stable


## 5. Predicting for a new member

A fitted pipeline accepts raw data with the same columns, and the encoding and scaling happen inside it.

In [17]:
new = pd.DataFrame({"age": [24], "tenure_years": [0.5], "tier": ["basic"], "channel": ["online"]})
if hasattr(pipe, "predict_proba"):
    print(f"Churn probability: {pipe.predict_proba(new)[0, 1]:.1%}")

Churn probability: 53.6%


## Wrap-up

Commit from VS Code (message: `Session 16: scikit-learn fundamentals`). The two traps to remember: alphabetical reference levels, and regularisation on by default.